# Stacked Target — Havar + Mo-100 + Cu

This notebook demonstrates multi-layer energy propagation through a realistic target assembly:

1. **Havar window** (25 um) — beam entrance foil, natural composition
2. **Enriched Mo-100 target** — exit energy 12 MeV
3. **Cu backing** (0.5 mm) — thermal management

The beam loses energy in each layer sequentially. HYRR tracks the energy, computes isotope production per layer, and reports per-layer and combined results.

## Setup

In [ ]:
from hyrr import (
    Beam, DataStore, Layer, TargetStack,
    compute_stack, result_summary, result_to_polars,
)
from hyrr.materials import resolve_element, resolve_formula
from hyrr.plotting import plot_activity_vs_time, plot_depth_profile

db = DataStore("../data/parquet")

## Define the beam and layers

**Havar** is a cobalt-based superalloy commonly used as a beam window. Approximate composition (mass fractions): Co 42%, Cr 20%, Ni 13%, W 3%, Mo 2%, Mn 2%, Fe 18%.

In [ ]:
beam = Beam(projectile="p", energy_MeV=16.0, current_mA=0.15)

# --- Layer 1: Havar window (25 um) ---
from hyrr.materials import resolve_isotopics

havar_composition = {
    "Co": 0.42, "Cr": 0.20, "Fe": 0.18, "Ni": 0.13,
    "W": 0.03, "Mo": 0.02, "Mn": 0.02,
}
havar_elements = resolve_isotopics(db, havar_composition, is_atom_fraction=False)

havar = Layer(
    density_g_cm3=8.3,
    elements=havar_elements,
    thickness_cm=25e-4,  # 25 um
)

# --- Layer 2: Enriched Mo-100 target ---
mo100 = resolve_element(db, "Mo", enrichment={100: 1.0})

mo_target = Layer(
    density_g_cm3=10.22,
    elements=[(mo100, 1.0)],
    energy_out_MeV=12.0,
)

# --- Layer 3: Cu backing (0.5 mm) ---
cu = resolve_element(db, "Cu")

cu_backing = Layer(
    density_g_cm3=8.96,
    elements=[(cu, 1.0)],
    thickness_cm=0.05,  # 0.5 mm
)

stack = TargetStack(
    beam=beam,
    layers=[havar, mo_target, cu_backing],
    irradiation_time_s=86400.0,
    cooling_time_s=86400.0,
    area_cm2=1.0,
)

print(f"Stack: {len(stack.layers)} layers")

## Run simulation

In [ ]:
result = compute_stack(db, stack)
print(result_summary(result))

## Energy loss through the stack

In [ ]:
layer_names = ["Havar window", "Mo-100 target", "Cu backing"]

for name, lr in zip(layer_names, result.layer_results):
    print(f"{name:>15s}:  E_in = {lr.energy_in:6.2f} MeV  ->  E_out = {lr.energy_out:6.2f} MeV  (dE = {lr.delta_E_MeV:.2f} MeV, heat = {lr.heat_kW:.3f} kW)")

## Per-layer results

In [ ]:
df = result_to_polars(result)
df.sort("activity_Bq", descending=True).head(20)

## Depth profiles — heat deposition per layer

In [ ]:
for name, lr in zip(layer_names, result.layer_results):
    if lr.depth_profile:
        fig = plot_depth_profile(lr, quantity="heat", title=f"Heat — {name}")
        fig.set_size_inches(8, 4)
        fig.show()

## Mo-100 layer — activity vs time

In [ ]:
# Mo-100 is layer index 1 (after Havar)
mo_lr = result.layer_results[1]
plot_activity_vs_time(mo_lr.isotope_results, top_n=8)

## Stopping power sources per layer

In [ ]:
for name, lr in zip(layer_names, result.layer_results):
    sources = {db.get_element_symbol(Z): src for Z, src in lr.stopping_power_sources.items()}
    print(f"{name}: {sources}")